# Modul 11: Metriken, Kreuzvalidierung, Suche und Erklärbarkeit

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Metriken und Suche, Merkmale erklären  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 125 bis 175 Minuten

    ## Überblick

    Sie bewerten Modelle mit mehreren Metriken, wählen passende Kreuzvalidierungssplitter und führen kleine leakage-sichere Hyperparametersuchen durch. Danach vergleichen Sie Merkmalsauswahl, Dimensionsreduktion und einfache globale Modellinterpretationen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_11A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_11B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Passende Metriken für Klassifikation und Regression auswählen.
- Leakage-sichere Kreuzvalidierung mit passenden Splittern durchführen.
- Hyperparameter mit kleinen Suchräumen optimieren und Ergebnisse dokumentieren.
- Merkmale innerhalb von scikit-learn-Pipelines konstruieren und auswählen.
- Dimensionsreduktion und Merkmalsauswahl für unterschiedliche Datenformen vergleichen.
- Modelle mit Koeffizienten, Importances und Abhängigkeitsplots interpretieren.

    ## Bewertete Fähigkeiten

    - Balanced Accuracy, Precision, Recall, F1, ROC-AUC und Average Precision
- StratifiedKFold, GroupKFold, TimeSeriesSplit und cross_validate
- GridSearchCV und RandomizedSearchCV mit Pipelines
- SelectKBest, PCA und polynomiale Merkmale
- Koeffizienten, Feature Importances und Partial Dependence

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.decomposition import PCA

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_metrics, y_metrics = make_classification(
    n_samples=650,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.88, 0.12],
    class_sep=1.0,
    flip_y=0.03,
    random_state=RANDOM_SEED,
)
feature_names_11 = [f"feature_{i:02d}" for i in range(X_metrics.shape[1])]
groups_11 = np.repeat(np.arange(130), 5)

X_reg_11, y_reg_11 = make_regression(
    n_samples=360,
    n_features=5,
    n_informative=4,
    noise=12.0,
    random_state=RANDOM_SEED,
)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Metriken, ROC und Precision-Recall vergleichen

    Erstellen Sie einen stratifizierten Train/Test-Split für `X_metrics`, `y_metrics` und trainieren Sie eine skalierte logistische Regression.

1. Berechnen Sie Accuracy, Balanced Accuracy, Precision, Recall, F1, ROC-AUC und Average Precision.
2. Erstellen Sie die Konfusionsmatrix und einen `classification_report`.
3. Zeichnen Sie ROC- und Precision-Recall-Kurve als getrennte Abbildungen.
4. Wiederholen Sie Precision, Recall und F1 mit Schwellenwert 0.30.
5. Begründen Sie, welche Metriken bei der ungleichen Klassenverteilung besonders informativ sind.

> **Hinweis:** Schwellenwertabhängige und schwellenwertfreie Metriken beantworten unterschiedliche Fragen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_metrics, y_metrics, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_metrics
)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Schwellenwertabhängige und schwellenwertfreie Metriken beantworten unterschiedliche Fragen.

## Aufgabe 2: Kreuzvalidierungssplitter passend einsetzen

    Vergleichen Sie vier Splitter:

- `KFold` für eine allgemeine Regression,
- `StratifiedKFold` für die ungleiche Klassifikation,
- `GroupKFold` mit `groups_11`,
- `TimeSeriesSplit` für die nach Index geordneten Regressionsdaten.

Verwenden Sie `cross_validate` mit mindestens zwei Metriken und speichern Sie Mittelwert und Standardabweichung der Validierungsergebnisse. Prüfen Sie bei `GroupKFold` ausdrücklich, dass keine Gruppe gleichzeitig in Train und Validierung liegt.

> **Hinweis:** Vorverarbeitung gehört in die Pipeline, damit sie innerhalb jedes Trainingsfolds neu gelernt wird.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Vorverarbeitung gehört in die Pipeline, damit sie innerhalb jedes Trainingsfolds neu gelernt wird.

## Aufgabe 3: Grid Search und Randomized Search dokumentieren

    Optimieren Sie eine skalierte logistische Regression auf `X_metrics`, `y_metrics` mit `StratifiedKFold` und der Zielmetrik `average_precision`.

1. Grid Search über `C=[0.05, 0.2, 1, 5]` und `class_weight=[None, "balanced"]`.
2. Randomized Search über dieselben Werte plus `solver=["liblinear", "lbfgs"]`, mit höchstens sechs Kandidaten.
3. Speichern Sie Rang, Mittelwert, Standardabweichung und Parameter der besten Ergebnisse in einer Tabelle.
4. Bewerten Sie den besten Grid-Search-Schätzer auf einem vorher zurückgehaltenen Testsatz.

> **Hinweis:** Der Testsatz darf weder Hyperparameterwahl noch Schwellenwertentscheidung beeinflussen.

In [ ]:
X_search_train, X_search_test, y_search_train, y_search_test = train_test_split(
    X_metrics, y_metrics, test_size=0.20,
    random_state=RANDOM_SEED, stratify=y_metrics
)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Der Testsatz darf weder Hyperparameterwahl noch Schwellenwertentscheidung beeinflussen.

## Aufgabe 4: Merkmalsauswahl, PCA und polynomiale Merkmale vergleichen

    Vergleichen Sie drei Pipelinevarianten mit stratifizierter Kreuzvalidierung:

1. Skalierung + logistische Regression auf allen Merkmalen.
2. Skalierung + `SelectKBest(f_classif, k=6)` + logistische Regression.
3. Skalierung + `PCA(n_components=6)` + logistische Regression.

Erstellen Sie zusätzlich für `X_reg_11` eine Ridge-Pipeline mit `PolynomialFeatures(degree=2, include_bias=False)` und vergleichen Sie deren CV-MAE mit einer linearen Ridge-Pipeline. Dokumentieren Sie Ausgabedimensionen und Metriken.

> **Hinweis:** Vergleichen Sie nicht nur den Score, sondern auch Ausgabedimension und Interpretierbarkeit.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Vergleichen Sie nicht nur den Score, sondern auch Ausgabedimension und Interpretierbarkeit.

## Aufgabe 5: Integrationsaufgabe: Koeffizienten, Importances und Partial Dependence

    Trainieren Sie auf einem festen Split:

- eine skalierte logistische Regression,
- einen begrenzten Random Forest.

1. Erstellen Sie Ranglisten aus absoluten LogReg-Koeffizienten und Forest-Importances.
2. Vergleichen Sie die Top-5-Merkmale beider Modelle.
3. Erstellen Sie einen Partial-Dependence-Plot für das wichtigste Forest-Merkmal.
4. Schreiben Sie eine kurze Erklärung, was jede Methode zeigt und welche Grenzen sie besitzt.

> **Hinweis:** Interpretieren Sie immer das konkrete Modell und seine Datenbasis, nicht vermeintliche Naturgesetze.

In [ ]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_metrics, y_metrics, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_metrics
)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Interpretieren Sie immer das konkrete Modell und seine Datenbasis, nicht vermeintliche Naturgesetze.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.